# 06 — Evaluate GaitTR

Loads checkpoint, extracts embeddings, computes Rank-1 for NM/BG/CL.

In [ ]:
from pathlib import Path
import os, random, time, gc, math
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt
from tqdm.notebook import tqdm

EXP_DIR = Path('/media/wadud/DriveUbuntu/GaitRecognition 2.0')
POSE_TAG = None  # None = auto-detect folder under data/poses with most .npz files
POSES_DIR = EXP_DIR / 'data' / 'poses'
SPLIT_DIR = EXP_DIR / 'data' / 'splits'
REPORT_DIR = EXP_DIR / 'data' / 'reports'
RESULT_DIR = EXP_DIR / 'results'
CHECKPOINT_DIR = EXP_DIR / 'checkpoints'
LOG_DIR = EXP_DIR / 'logs'
for d in [SPLIT_DIR, REPORT_DIR, RESULT_DIR, CHECKPOINT_DIR, LOG_DIR]:
    d.mkdir(parents=True, exist_ok=True)

def detect_pose_root(poses_dir=POSES_DIR, pose_tag=POSE_TAG):
    if pose_tag is not None:
        root = poses_dir / pose_tag
        if not root.exists():
            raise FileNotFoundError(root)
        return root
    candidates = [p for p in poses_dir.iterdir() if p.is_dir()] if poses_dir.exists() else []
    if not candidates:
        raise FileNotFoundError(f'No pose folders under {poses_dir}')
    counts = sorted([(len(list(p.rglob('*.npz'))), p) for p in candidates], reverse=True, key=lambda x: x[0])
    return counts[0][1]

POSE_ROOT = detect_pose_root()
print('EXP_DIR  :', EXP_DIR)
print('POSE_ROOT:', POSE_ROOT)

SPLIT_NAME='LT'; SEQ_LEN=60; STRIDE=30; BATCH_CLIPS=64; EXCLUDE_IDENTICAL_VIEW=True; CHECKPOINT_PATH=None
CHECKPOINT_DIR.mkdir(parents=True,exist_ok=True); RESULT_DIR.mkdir(parents=True,exist_ok=True)
device=torch.device('cuda' if torch.cuda.is_available() else 'cpu'); print('device:',device)
if CHECKPOINT_PATH is None:
    cand=CHECKPOINT_DIR/f'gaittr_{SPLIT_NAME}_last.pth'
    CHECKPOINT_PATH=cand if cand.exists() else sorted(CHECKPOINT_DIR.glob('*.pth'))[-1]
CHECKPOINT_PATH=Path(CHECKPOINT_PATH); print('checkpoint:',CHECKPOINT_PATH)

In [ ]:
COCO_PARENTS = np.array([0,0,0,1,2,0,0,5,6,7,8,5,6,11,12,13,14], dtype=np.int64)
LEFT_RIGHT_PAIRS = [(1,2),(3,4),(5,6),(7,8),(9,10),(11,12),(13,14),(15,16)]

def crop_or_pad_sequence(X, seq_len=60, random_crop=True):
    T = X.shape[0]
    if T == seq_len:
        return X
    if T > seq_len:
        start = np.random.randint(0, T-seq_len+1) if random_crop else max(0, (T-seq_len)//2)
        return X[start:start+seq_len]
    pad = np.repeat(X[-1:], seq_len-T, axis=0)
    return np.concatenate([X, pad], axis=0)

def swap_left_right(X):
    X = X.copy()
    for l, r in LEFT_RIGHT_PAIRS:
        X[:, [l, r], :] = X[:, [r, l], :]
    return X

def augment_skeleton(X):
    if np.random.rand() < 0.5:
        X = swap_left_right(X)
    X = X + np.random.normal(0, 0.005, size=X.shape).astype(np.float32)
    X = X + np.random.normal(0, 0.01, size=(1,1,2)).astype(np.float32)
    return X

def build_gaittr_features(X):
    X = X.astype(np.float32)
    assert X.ndim == 3 and X.shape[1:] == (17,2), f'Bad skeleton shape: {X.shape}'
    joint = X.copy()
    joint_rel = X - X[:, 0:1, :]
    v1 = np.zeros_like(X); v1[:-1] = X[1:] - X[:-1]
    v2 = np.zeros_like(X); v2[:-2] = X[2:] - X[:-2]
    bone = X - X[:, COCO_PARENTS, :]
    feat = np.concatenate([joint, joint_rel, v1, v2, bone], axis=-1)
    return feat.transpose(2,0,1).astype(np.float32)  # 10,T,17

In [ ]:
class TCN(nn.Module):
    def __init__(self, in_channels, out_channels, temporal_kernel=9, dropout=0.1):
        super().__init__(); pad = temporal_kernel // 2
        self.net = nn.Sequential(
            nn.Conv2d(in_channels, out_channels, kernel_size=(temporal_kernel,1), padding=(pad,0), bias=False),
            nn.Dropout(dropout), nn.Mish(), nn.BatchNorm2d(out_channels)
        )
    def forward(self, x): return self.net(x)

class SpatialTransformer(nn.Module):
    def __init__(self, channels, num_heads=8, dropout=0.1):
        super().__init__(); assert channels % num_heads == 0
        self.norm = nn.LayerNorm(channels)
        self.attn = nn.MultiheadAttention(channels, num_heads, dropout=dropout, batch_first=True)
        self.proj = nn.Linear(channels, channels)
        self.act = nn.Mish(); self.bn = nn.BatchNorm2d(channels)
    def forward(self, x):
        B,C,T,V = x.shape
        tok = x.permute(0,2,3,1).contiguous().view(B*T, V, C)
        tok = self.norm(tok)
        out, _ = self.attn(tok, tok, tok, need_weights=False)
        out = self.proj(out).view(B,T,V,C).permute(0,3,1,2).contiguous()
        return self.bn(self.act(out))

class TCNSTBlock(nn.Module):
    def __init__(self, in_channels, out_channels, num_heads=8, temporal_kernel=9, dropout=0.1):
        super().__init__()
        self.tcn = TCN(in_channels, out_channels, temporal_kernel, dropout)
        self.st = SpatialTransformer(out_channels, num_heads, dropout)
        self.res = nn.Identity() if in_channels == out_channels else nn.Sequential(
            nn.Conv2d(in_channels, out_channels, kernel_size=1, bias=False), nn.Mish(), nn.BatchNorm2d(out_channels)
        )
    def forward(self, x): return self.st(self.tcn(x)) + self.res(x)

class GaitTR(nn.Module):
    def __init__(self, in_channels=10, embedding_dim=128, channels=(64,64,128,256), num_heads=8, temporal_kernel=9, dropout=0.1):
        super().__init__(); self.data_bn = nn.BatchNorm2d(in_channels)
        blocks=[]; prev=in_channels
        for ch in channels:
            blocks.append(TCNSTBlock(prev, ch, num_heads, temporal_kernel, dropout)); prev=ch
        self.blocks = nn.Sequential(*blocks); self.fc = nn.Linear(channels[-1], embedding_dim, bias=False)
    def forward(self, x):
        x = self.data_bn(x); x = self.blocks(x); x = x.mean(dim=(2,3))
        return F.normalize(self.fc(x), p=2, dim=1)

class BatchHardTripletLoss(nn.Module):
    def __init__(self, margin=0.3): super().__init__(); self.margin=margin
    def forward(self, embeddings, labels):
        dist = torch.cdist(embeddings, embeddings, p=2)
        labels = labels.view(-1,1); same = labels.eq(labels.t()); diff = ~same
        eye = torch.eye(labels.size(0), device=labels.device, dtype=torch.bool); same = same & ~eye
        hp = dist.masked_fill(~same, -1e9).max(dim=1)[0]
        hn = dist.masked_fill(~diff, 1e9).min(dim=1)[0]
        loss = F.relu(hp - hn + self.margin); valid = hp > -1e8
        if valid.sum() == 0: return torch.tensor(0.0, device=embeddings.device, requires_grad=True)
        return loss[valid].mean()

In [ ]:
ckpt=torch.load(CHECKPOINT_PATH,map_location=device); cfg=ckpt.get('config',{})
channels=tuple(cfg.get('channels',(64,64,128,256))); emb_dim=int(cfg.get('embedding_dim',128))
model=GaitTR(embedding_dim=emb_dim,channels=channels).to(device); model.load_state_dict(ckpt['model']); model.eval()
print('loaded step:',ckpt.get('step','unknown'),'channels:',channels)

def make_sliding_clips(X,clip_len=60,stride=30):
    T=X.shape[0]
    if T<=clip_len: return [crop_or_pad_sequence(X,clip_len,random_crop=False)]
    clips=[X[s:s+clip_len] for s in range(0,T-clip_len+1,stride)]
    return clips if clips else [crop_or_pad_sequence(X,clip_len,random_crop=False)]

@torch.no_grad()
def extract_video_embedding(pose_path):
    data=np.load(pose_path); X=data['keypoints_norm_filled'].astype(np.float32)
    feats=np.stack([build_gaittr_features(c) for c in make_sliding_clips(X,SEQ_LEN,STRIDE)],axis=0)
    outs=[]
    for s in range(0,len(feats),BATCH_CLIPS):
        x=torch.from_numpy(feats[s:s+BATCH_CLIPS]).float().to(device); outs.append(model(x).cpu())
    emb=torch.cat(outs,dim=0).mean(dim=0); return F.normalize(emb,p=2,dim=0)

def extract_df(df,name):
    metas=[]; embs=[]
    for _,row in tqdm(df.iterrows(),total=len(df),desc=name):
        embs.append(extract_video_embedding(row.pose_path)); metas.append(row.to_dict())
    return pd.DataFrame(metas), torch.stack(embs,dim=0)

In [ ]:
paths={
 'gallery':SPLIT_DIR/f'gallery_{SPLIT_NAME}.csv',
 'nm':SPLIT_DIR/f'probe_{SPLIT_NAME}_nm.csv',
 'bg':SPLIT_DIR/f'probe_{SPLIT_NAME}_bg.csv',
 'cl':SPLIT_DIR/f'probe_{SPLIT_NAME}_cl.csv'}
for p in paths.values(): assert p.exists(),p
df_gallery=pd.read_csv(paths['gallery']); df_nm=pd.read_csv(paths['nm']); df_bg=pd.read_csv(paths['bg']); df_cl=pd.read_csv(paths['cl'])
print(len(df_gallery),len(df_nm),len(df_bg),len(df_cl))
g_meta,g_emb=extract_df(df_gallery,'gallery')
nm_meta,nm_emb=extract_df(df_nm,'probe_nm')
bg_meta,bg_emb=extract_df(df_bg,'probe_bg')
cl_meta,cl_emb=extract_df(df_cl,'probe_cl')
torch.save({'meta':g_meta,'embeddings':g_emb},RESULT_DIR/f'gallery_{SPLIT_NAME}_embeddings.pt')
torch.save({'meta':nm_meta,'embeddings':nm_emb},RESULT_DIR/f'probe_{SPLIT_NAME}_nm_embeddings.pt')
torch.save({'meta':bg_meta,'embeddings':bg_emb},RESULT_DIR/f'probe_{SPLIT_NAME}_bg_embeddings.pt')
torch.save({'meta':cl_meta,'embeddings':cl_emb},RESULT_DIR/f'probe_{SPLIT_NAME}_cl_embeddings.pt')

In [ ]:
def evaluate(g_meta,g_emb,p_meta,p_emb,cond):
    g_sub=g_meta.subject.astype(str).tolist(); g_view=g_meta.view.astype(str).tolist(); rows=[]
    for i in tqdm(range(len(p_meta)),desc=f'eval {cond}'):
        ps=str(p_meta.iloc[i].subject); pv=str(p_meta.iloc[i].view)
        d=torch.norm(g_emb-p_emb[i].unsqueeze(0),p=2,dim=1)
        if EXCLUDE_IDENTICAL_VIEW:
            mask=torch.tensor([v!=pv for v in g_view],dtype=torch.bool); d[~mask]=float('inf')
        bi=int(torch.argmin(d).item()); pred=g_sub[bi]; ok=int(pred==ps)
        rows.append({'condition':cond,'subject':ps,'view':pv,'seq':str(p_meta.iloc[i].seq),'pose_path':p_meta.iloc[i].pose_path,
                     'pred_subject':pred,'pred_view':g_view[bi],'distance':float(d[bi].item()),'correct':ok})
    df=pd.DataFrame(rows); return df.correct.mean(),df
acc_nm,res_nm=evaluate(g_meta,g_emb,nm_meta,nm_emb,'NM')
acc_bg,res_bg=evaluate(g_meta,g_emb,bg_meta,bg_emb,'BG')
acc_cl,res_cl=evaluate(g_meta,g_emb,cl_meta,cl_emb,'CL')
res=pd.concat([res_nm,res_bg,res_cl],ignore_index=True)
sumdf=pd.DataFrame([{'condition':'NM','rank1':acc_nm,'num_probe':len(res_nm)},{'condition':'BG','rank1':acc_bg,'num_probe':len(res_bg)},{'condition':'CL','rank1':acc_cl,'num_probe':len(res_cl)}])
res.to_csv(RESULT_DIR/f'rank1_{SPLIT_NAME}_all_probe_results.csv',index=False); sumdf.to_csv(RESULT_DIR/f'rank1_{SPLIT_NAME}_summary.csv',index=False)
per_view=res.groupby(['condition','view'])['correct'].mean().reset_index().rename(columns={'correct':'rank1'}); per_view.to_csv(RESULT_DIR/f'rank1_{SPLIT_NAME}_per_view.csv',index=False)
display(sumdf); display(per_view)